# Evaluate all VNC Cell Types
**Aim:** We want to find the neurons associated with the VNC (Ventral Nerve Cord) and identify whether the neuron is likely to express one of the 6 neuro transmitters trained by the model.

**Ideal Output:**
| cell_type | root_id | gaba_pos | ach_pos | glut_pos | oct_pos | ser_pos | da_pos | cotransmission_pos |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| the cell type | the codex neuron identifier | true/false  | true/false | true/false | true/false | true/false | true/false | true/false |
| ... | ... | ...  | ... | ... | ... | ... | ... | ... |

## Base functions
These are to be taken from the previous notebooks so that we can loop through everything programatically

### Load libraries

In [2]:
#libraries
import os
import numpy as np
import pandas as pd
from decouple import config, Config, RepositoryEnv
from caveclient import CAVEclient
from fafbseg import flywire
from getpass import getpass
from sklearn.cluster import estimate_bandwidth, mean_shift
import seaborn as sns
import matplotlib.pyplot as plt
import navis
import datetime
import signal
import time

/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Client connections
We check if the tokens for both the CAVE client and the flywire client are present and are running.

In [3]:
# Get variables from the .env file
ENV_PATH = "../.env"
config = Config(RepositoryEnv(ENV_PATH))

### CAV Client ###
# Get the CAVE_AUTH_TOKEN
cave_token = config("CAVE_AUTH_TOKEN", default=None)
if not cave_token:
    print("No CAVE token found in the .env file.")
    temp_token = getpass("Token not found in the .env file. Please enter your CAVE token (if you don't have one leave if blank and instructions will appear): ")
    if temp_token:
        cave_token = temp_token
    else:
        print("No CAVE token entered. Follow the information below to get a token.")
        CAVEclient.auth.get_new_token()
if cave_token:
    # Initialize the CAVE client
    client = CAVEclient('flywire_fafb_public')
    auth = client.auth
    tk = auth.token
    if not tk:
        print("Adding the token to your account. You only need to do this once.")
        auth.save_token(cave_token)
    else:
        print("CAVE token already exists in your account. No need to add it again.")

### Get the FLYWIRE_AUTH_TOKEN ###
flywire_token = config("FLYWIRE_AUTH_TOKEN", default=None)
tk = flywire.get_chunkedgraph_secret()
if not tk:
    print("No FLYWIRE token saved. Setting it with the value from the .env file.")
    if not flywire_token:
        flywire_token = getpass("Token not found in the .env file. Please enter your FLYWIRE token (if you don't have one leave if blank and instructions will appear): ")
        if not flywire_token:
            print("No FLYWIRE token entered. Follow the information below to get a token.")
            flywire.get_new_token()
        else:
            print("Setting the FLYWIRE token with the value from the .env file.")
            #Save the secret
            flywire.set_chunkedgraph_secret(flywire_token)
else:
    print("FLYWIRE token already exists in your account. No need to add it again.")

CAVE token already exists in your account. No need to add it again.
FLYWIRE token already exists in your account. No need to add it again.


## Check to see if it is possible to sort data by input or output regions

In [4]:
all_tables = client.materialize.get_tables()
for table in all_tables:
    print(f"Evaluating table: {table}")
    #Display the table metadata
    table_metadata = client.materialize.get_table_metadata(table)
    print(f"Table metadata: {table_metadata}")
    #Display the first few rows of the table
    table_df = client.materialize.query_table(table, limit=5)
    print(f"{table} head")
    display(table_df)

    #Print column names
    print(f"Column names: {table_df.columns.tolist()}")
    

Evaluating table: hierarchical_neuron_annotations
Table metadata: {'created': '2023-06-19T21:52:24.506754', 'id': 7468, 'table_name': 'hierarchical_neuron_annotations', 'aligned_volume': 'fafb_seung_alignment_v0', 'schema': 'cell_type_reference', 'valid': True, 'schema_type': 'cell_type_reference', 'user_id': '2', 'description': "\nThis table contains the hierarchical class annotations introduced with the FlyWire 630 release. \n\nThe annotations were produced by Schlegel et al. and Schlegel et al., 2023 should be credited for their use.\n\nThe hierarchy goes as follows:\nflow > super_class > cell_class > cell_sub_class > cell_type\n\nNot all neurons were annotated with a fine grained level.\n [Note: This table 'hierarchical_neuron_annotations' will update the 'target_id' foreign_key when updates are made to the 'proofread_neurons' table] ", 'notice_text': None, 'reference_table': 'proofread_neurons', 'flat_segmentation_source': None, 'write_permission': 'PRIVATE', 'read_permission': 'P

201 - "Limited query to 5 rows


hierarchical_neuron_annotations head


,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pt_supervoxel_id,pt_root_id,pt_position
0,601615,2023-06-19 23:31:33.449997+00:00,t,1,super_class,central,1,2023-06-19 06:43:32.562416+00:00,t,78253067652813181,720575940628857210,"[443342, 203965, 157450]"
1,473637,2023-06-19 23:26:01.973320+00:00,t,1,flow,intrinsic,1,2023-06-19 06:43:32.562416+00:00,t,78253067652813181,720575940628857210,"[443342, 203965, 157450]"
2,879934,2023-09-30 05:09:51.441140+00:00,t,2,cell_type,CB0924,2,2023-06-19 06:43:32.563301+00:00,t,82053323167374089,720575940626838909,"[664417, 227538, 77011]"
3,601616,2023-06-19 23:31:33.451271+00:00,t,2,super_class,central,2,2023-06-19 06:43:32.563301+00:00,t,82053323167374089,720575940626838909,"[664417, 227538, 77011]"
4,473638,2023-06-19 23:26:01.974049+00:00,t,2,flow,intrinsic,2,2023-06-19 06:43:32.563301+00:00,t,82053323167374089,720575940626838909,"[664417, 227538, 77011]"


Column names: ['id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type', 'id_ref', 'created_ref', 'valid_ref', 'pt_supervoxel_id', 'pt_root_id', 'pt_position']
Evaluating table: neuron_information_v2
Table metadata: {'valid': True, 'id': 7469, 'aligned_volume': 'fafb_seung_alignment_v0', 'schema': 'bound_tag_user', 'created': '2022-02-07T08:25:46.532690', 'table_name': 'neuron_information_v2', 'schema_type': 'bound_tag_user', 'user_id': '2024', 'description': "\nNeuron Information v2\n\nThis table contains proofreading information from many contributors in FlyWire.\n\nNeuron information was provided by users. Each entry is associated with a user through the `user_id` column and that user should be credited for that contribution according to FlyWire's principles.\nIn some cases, multiple entries will match to the same (current) `pt_root_id`. In such cases multiple users contributed information for this neuron. \n", 'notice_text': None, 'reference_table': None, 'flat_s

...

Column names: ['id', 'created', 'superceded_id', 'valid', 'pt_supervoxel_id', 'pt_root_id', 'pt_position']
Evaluating table: fly_synapses_neuropil_v6
Table metadata: {'created': '2024-02-14T00:31:05.415023', 'valid': True, 'table_name': 'fly_synapses_neuropil_v6', 'id': 7473, 'schema': 'fly_neuropil', 'aligned_volume': 'fafb_seung_alignment_v0', 'schema_type': 'fly_neuropil', 'user_id': '2', 'description': "\nThis reference table contains neuropil assignments for all synapses. \n\nThis is the left/right inverted version of fly_synapses_neuropil_v2. \nWe found that FAFB's imagery had been accidentily flipped during the initial processing. These neuropil annotations reflect that by inverting the Left/Right assignments.\n\nneuropil layer for neuroglancer: precomputed://gs://flywire_neuropil_meshes/neuropils/neuropil_mesh_v141_v6\n\nThe neuropils originated from Chiang et al., 2011 (FlyCircuit) were imported by Greg Jefferis and refined by Philipp Schlegel. Philipp Schlegel generated addit

201 - "Limited query to 5 rows


fly_synapses_neuropil_v6 head


,id_ref,created_ref,valid_ref,connection_score,cleft_score,gaba,ach,glut,oct,ser,...,pre_pt_root_id,post_pt_supervoxel_id,post_pt_root_id,id,created,valid,target_id,neuropil,pre_pt_position,post_pt_position
0,0,2021-03-09 20:14:58.183080+00:00,t,7.635252,0,0.248573,0.072179,0.057031,0.019089,0.112843,...,720575940630479697,76916748439628340,720575940605404546,1,2024-02-14 00:39:26.078645+00:00,t,0,AVLP_L,"[366196, 244760, 73920]","[366296, 244740, 73960]"
1,1,2021-03-09 20:14:58.183080+00:00,t,15.259161,0,0.169801,0.312128,0.068937,0.044355,0.268214,...,720575940630479697,76916748439614229,720575940616174657,2,2024-02-14 00:39:26.078645+00:00,t,1,AVLP_L,"[364864, 245640, 73960]","[364708, 245588, 73960]"
2,2,2021-03-09 20:14:58.183080+00:00,t,10.514650,0,0.616433,0.223582,0.047966,0.052839,0.028816,...,720575940616174657,76916748439623112,720575940545890960,3,2024-02-14 00:39:26.078645+00:00,t,2,AVLP_L,"[364564, 244904, 73960]","[364680, 244840, 74000]"
3,3,2021-03-09 20:14:58.183080+00:00,t,71.585907,0,0.486084,0.209755,0.246938,0.000980,0.027569,...,720575940630479697,76916748439633329,720575940633787309,4,2024-02-14 00:39:26.078645+00:00,t,3,AVLP_L,"[364600, 244468, 74200]","[364552, 244344, 74200]"
4,4,2021-03-09 20:14:58.183080+00:00,t,21.044254,0,0.338186,0.584693,0.027468,0.000612,0.023012,...,720575940630906435,76916748439632348,720575940630479697,5,2024-02-14 00:39:26.078645+00:00,t,4,AVLP_L,"[364952, 244764, 74240]","[364860, 244796, 74200]"


Column names: ['id_ref', 'created_ref', 'valid_ref', 'connection_score', 'cleft_score', 'gaba', 'ach', 'glut', 'oct', 'ser', 'da', 'valid_nt', 'pre_pt_supervoxel_id', 'pre_pt_root_id', 'post_pt_supervoxel_id', 'post_pt_root_id', 'id', 'created', 'valid', 'target_id', 'neuropil', 'pre_pt_position', 'post_pt_position']


So it seems like the only possible option is `fly_synapses_neuropil_v6` from which I will need the `pre_pt_root_id` and `post_pt_root_id` to find the `root_id` of the neurons I want. 
* My assumption is that the pre_pt_root_id neurons will mostly be projecting neurons from the brain to the VNC neuropil areas
* The post_pt_root_id neurons will either be local, or ascending neurons back to the brain

However, this might be tricky as I will need to find the names of all the neuropils.

In [5]:
# Find all the names of the neuropils
# np_df = client.materialize.query_table("fly_synapses_neuropil_v6", limit=0)  # schema only
neuropils = client.materialize.query_table(
    "fly_synapses_neuropil_v6",
    select_columns=['neuropil','pre_pt_root_id','post_pt_root_id'],
)
neuropil_list = neuropils['neuropil'].unique().tolist()
# Display the neuropils
print("Neuropils found in the fly_synapses_neuropil_v6 table:")
print(neuropil_list)


Select_columns is deprecated for join queries, please use select_column_map a dictionary which is more explicit about what columns to select from what tables. This query result will attempt to select the first column it finds of this name in any table, but if there are more than one such column it will not select both. Upgrade caveclient to >=5.0.0 ., 201 - "Limited query to 500000 rows


Neuropils found in the fly_synapses_neuropil_v6 table:
['AVLP_L', 'AL_L', 'AL_R', None, 'AVLP_R', 'LAL_R', 'LA_R', 'MB_ML_L', 'CRE_R', 'CRE_L', 'MB_ML_R', 'SIP_R', 'AOTU_R', 'SLP_R', 'MB_VL_L', 'MB_VL_R', 'MB_PED_L', 'AOTU_L', 'EB', 'SMP_L', 'SMP_R', 'SIP_L', 'GNG', 'LAL_L', 'PVLP_R', 'GA_L', 'GA_R', 'MB_PED_R', 'PRW', 'FLA_R', 'AMMC_R', 'SAD', 'AMMC_L', 'FLA_L']


All of the neuropils are in the brain so this won't work

### Trial by regex searches
See if we can get all the required cell types by regular expressions

In [6]:
#Build a regex to get all neurons that start with DN
expression = "^DN.*"

descending_neuron_df = client.materialize.query_table(
        "hierarchical_neuron_annotations",
        filter_in_dict={'classification_system': ["cell_type"]},
        filter_regex_dict={'cell_type': expression}
    )
print(f"{len(descending_neuron_df['cell_type'].unique().tolist())} unique descending neurons found.")
descending_neuron_df.head(10)

442 unique descending neurons found.


,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pt_supervoxel_id,pt_root_id,pt_position
0,746710,2023-06-22 03:47:24.551288+00:00,t,32436,cell_type,DNge038,32436,2023-06-19 06:44:57.894846+00:00,t,79803172889271355,720575940637603038,"[532826, 322732, 149703]"
1,894164,2023-09-30 05:09:51.441140+00:00,t,32450,cell_type,DNb08_a,32450,2023-06-19 06:44:57.908210+00:00,t,78817117050795308,720575940630334658,"[477430, 269770, 138461]"
2,746707,2023-06-22 03:47:24.549044+00:00,t,32413,cell_type,DNpe032,32413,2023-06-19 06:44:57.859444+00:00,t,80575030118899801,720575940645693476,"[578466, 194893, 164945]"
3,746713,2023-06-22 03:47:24.554034+00:00,t,32489,cell_type,DNge004,32489,2023-06-19 06:44:57.951367+00:00,t,79380823052496309,720575940614352278,"[508698, 314891, 175063]"
4,746693,2023-06-22 03:47:24.537271+00:00,t,32280,cell_type,DNpe037,32280,2023-06-19 06:44:57.691789+00:00,t,78464173885017718,720575940623274966,"[457960, 204269, 147915]"
5,894089,2023-09-30 05:09:51.441140+00:00,t,32286,cell_type,DNp57,32286,2023-06-19 06:44:57.696588+00:00,t,78534680135178696,720575940646814340,"[460511, 213826, 165043]"
6,746695,2023-06-22 03:47:24.538655+00:00,t,32310,cell_type,DNge001,32310,2023-06-19 06:44:57.715405+00:00,t,80226003829806991,720575940622632084,"[557259, 359958, 155327]"
7,742998,2023-06-22 03:47:21.405372+00:00,t,420,cell_type,DNa06,420,2023-06-19 06:43:33.007956+00:00,t,80646635813825415,720575940640325429,"[582034, 267107, 169294]"
8,880113,2023-09-30 05:09:51.441140+00:00,t,426,cell_type,DNp41,426,2023-06-19 06:43:33.012821+00:00,t,80575305063955292,720575940632836675,"[577627, 208278, 185522]"
9,880111,2023-09-30 05:09:51.441140+00:00,t,418,cell_type,DNg98,418,2023-06-19 06:43:33.006400+00:00,t,79732529200258270,720575940623338281,"[531338, 306308, 133138]"


In [7]:
# Now get the Ascending Neurons
expression = ".*AN.*" #NB AN is not guaranteed to be at the begining or end

test_df = client.materialize.live_query(
        "hierarchical_neuron_annotations",
        datetime.datetime.now(),
        # filter_in_dict={'classification_system': ["cell_type"]},
        filter_regex_dict={'cell_type': expression},
    )
test_df

,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pt_supervoxel_id,pt_root_id,pt_position
0,638523,2023-06-19 23:33:22.608948+00:00,t,411,cell_class,DAN,411,2023-06-19 06:43:33.000464+00:00,t,79800561280835809,720575940630166267,"[533226, 167448, 71485]"
1,638511,2023-06-19 23:33:22.600695+00:00,t,32930,cell_class,DAN,32930,2023-06-19 06:44:58.392801+00:00,t,80082104909960188,720575940617242841,"[549645, 173555, 51077]"
2,633510,2023-06-19 23:33:09.173029+00:00,t,39357,cell_class,AN,39357,2023-06-19 06:45:19.942865+00:00,t,78112879852967701,720575940604865312,"[436177, 237472, 130803]"
3,638588,2023-06-19 23:33:22.696612+00:00,t,1495,cell_class,DAN,1495,2023-06-19 06:43:34.063294+00:00,t,78533442916191555,720575940629235047,"[460427, 141380, 85218]"
4,634395,2023-06-19 23:33:10.092873+00:00,t,40709,cell_class,AN,40709,2023-06-19 06:45:21.243811+00:00,t,80154810317925338,720575940623064909,"[553582, 312250, 120200]"
...,...,...,...,...,...,...,...,...,...,...,...,...
2688,638541,2023-06-19 23:33:22.622465+00:00,t,37019,cell_class,DAN,37019,2023-06-19 06:45:10.456484+00:00,t,80011598793633758,720575940627390507,"[547774, 165606, 63074]"
2689,638538,2023-06-19 23:33:22.619471+00:00,t,37020,cell_class,DAN,37020,2023-06-19 06:45:10.457302+00:00,t,80011598793704890,720575940638940469,"[546193, 165984, 64961]"
2690,638525,2023-06-19 23:33:22.610568+00:00,t,37021,cell_class,DAN,37021,2023-06-19 06:45:10.458457+00:00,t,79941504927396143,720575940622100774,"[541338, 179555, 64293]"
2691,638536,2023-06-19 23:33:22.617992+00:00,t,37023,cell_class,DAN,37023,2023-06-19 06:45:10.461055+00:00,t,80011736232629225,720575940624859725,"[546079, 172065, 64100]"


In [8]:
# Now get the Ascending Neurons
expression = ".*VUM.*" #NB AN is not guaranteed to be at the begining or end

test_df = client.materialize.live_query(
        "hierarchical_neuron_annotations",
        datetime.datetime.now(),
        filter_in_dict={'classification_system': ["cell_type"]},
        filter_regex_dict={'cell_type': expression},
    )
test_df

,id_ref,created_ref,valid_ref,pt_supervoxel_id,pt_root_id,id,created,valid,target_id,classification_system,cell_type,pt_position
0,37754,2023-06-19 06:45:18.611699+00:00,t,80223942111465016,720575940624394790,747458,2023-06-22 03:47:31.319815+00:00,t,37754,cell_type,OA-VUMa8,"[556943, 236322, 119119]"
1,41259,2023-06-19 06:45:21.751063+00:00,t,80013316982398665,720575940623164515,897225,2023-09-30 05:09:51.441140+00:00,t,41259,cell_type,OA-VUMa3,"[544995, 268070, 140504]"
2,117898,2023-06-19 06:48:46.481355+00:00,t,79379654754556021,720575940630460919,897226,2023-09-30 05:09:51.441140+00:00,t,117898,cell_type,OA-VUMa4,"[507835, 244895, 162172]"
3,117901,2023-06-19 06:48:46.485588+00:00,t,79732597919544690,720575940621875821,747709,2023-06-22 03:47:31.501447+00:00,t,117901,cell_type,OA-VUMx1,"[529327, 311144, 128330]"
4,139,2023-06-19 06:43:32.697721+00:00,t,79450504534925928,720575940626051514,879994,2023-09-30 05:09:51.441140+00:00,t,139,cell_type,OA-VUMa3,"[513643, 274981, 157727]"
5,2500,2023-06-19 06:43:35.304469+00:00,t,80153848379332050,720575940636479668,881037,2023-09-30 05:09:51.441140+00:00,t,2500,cell_type,OA-VUMa4,"[553057, 254133, 158155]"
6,22671,2023-06-19 06:44:32.870793+00:00,t,79732391627226387,720575940610708430,745689,2023-06-22 03:47:23.747503+00:00,t,22671,cell_type,OA-VUMx2,"[529844, 300755, 97704]"
7,35911,2023-06-19 06:45:09.092073+00:00,t,79518467828564907,720575940621662332,747293,2023-06-22 03:47:31.201138+00:00,t,35911,cell_type,OA-VUMx3,"[518256, 131954, 63246]"


### Check the "cell_class" groups
Perhaps I can get more data this way

In [9]:
# Now get the Ascending Neurons
# expression = ".*VUM.*" #NB AN is not guaranteed to be at the begining or end

test_df = client.materialize.live_query(
        "hierarchical_neuron_annotations",
        datetime.datetime.now(),
        filter_in_dict={'classification_system': ["cell_class"]},
        # filter_regex_dict={'cell_type': expression},
    )
test_df

,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pt_supervoxel_id,pt_root_id,pt_position
0,637236,2023-06-19 23:33:21.360472+00:00,t,32414,cell_class,CX,32414,2023-06-19 06:44:57.863498+00:00,t,79871411195357919,720575940620919646,"[538210, 198330, 105601]"
1,637235,2023-06-19 23:33:21.359470+00:00,t,32418,cell_class,CX,32418,2023-06-19 06:44:57.867936+00:00,t,79589661206805006,720575940624783287,"[523906, 181893, 72760]"
2,637234,2023-06-19 23:33:21.358542+00:00,t,32419,cell_class,CX,32419,2023-06-19 06:44:57.869497+00:00,t,79589661207074646,720575940630755276,"[523523, 181919, 80829]"
3,641830,2023-06-19 23:33:25.652976+00:00,t,32425,cell_class,Kenyon_Cell,32425,2023-06-19 06:44:57.877806+00:00,t,81911349130846298,720575940625290003,"[655290, 153943, 192032]"
4,637232,2023-06-19 23:33:21.357104+00:00,t,32426,cell_class,CX,32426,2023-06-19 06:44:57.879544+00:00,t,79589729993523306,720575940635992718,"[520015, 186587, 97417]"
...,...,...,...,...,...,...,...,...,...,...,...,...
107399,843626,2023-09-30 05:09:51.441140+00:00,t,104828,cell_class,ME,104828,2023-06-19 06:48:15.001237+00:00,t,83952592400822837,720575940626100238,"[776911, 183792, 174548]"
107400,843605,2023-09-30 05:09:51.441140+00:00,t,104809,cell_class,ME>LO.LOP,104809,2023-06-19 06:48:14.983796+00:00,t,84798048190170122,720575940626915216,"[824625, 246096, 194229]"
107401,820708,2023-09-30 05:09:51.441140+00:00,t,83362,cell_class,ME>LO.LOP,83362,2023-06-19 06:47:20.709997+00:00,t,75298954786869572,720575940645731620,"[272447, 288010, 156593]"
107402,843616,2023-09-30 05:09:51.441140+00:00,t,104819,cell_class,ME>LO.LOP,104819,2023-06-19 06:48:14.992064+00:00,t,84658822530652061,720575940629075755,"[815896, 335911, 204215]"


In [10]:
cell_types = test_df['cell_type'].unique().tolist()
print(f"Found {len(cell_types)} unique cell types in the hierarchical_neuron_annotations table.")
print("Cell types found:")
for cell_type in cell_types:
    print(cell_type)

Found 48 unique cell types in the hierarchical_neuron_annotations table.
Cell types found:
CX
Kenyon_Cell
ALPN
LO
bilateral
ME
ME>LOP
ME>LO
LO>LOP
ME>LA
LA>ME
ME>LO.LOP
ALLN
olfactory
DAN
MBON
ME>LOP.LO
LOP>ME.LO
LOP
LOP>LO.ME
LOP>LO
LA
AN
visual
TuBu
LHCENT
ALIN
mAL
LHLN
ME.LO
mechanosensory
ME.LO.LOP
LO>ME
hygrosensory
pars_lateralis
unknown_sensory
LO.LOP
ocellar
optic_lobes
pars_intercerebralis
ME.LOP
gustatory
ALON
MBIN
thermosensory
clock
LOP>ME
TPN


### Explore the flow classification system

In [11]:
test_df = client.materialize.live_query(
        "hierarchical_neuron_annotations",
        datetime.datetime.now(),
        filter_in_dict={'classification_system': ["flow"]},
        # filter_regex_dict={'cell_type': expression},
    )
test_df

,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pt_supervoxel_id,pt_root_id,pt_position
0,760326,2023-09-30 05:09:51.441140+00:00,t,135,flow,efferent,135,2023-06-19 06:43:32.694340+00:00,t,78536398189730110,720575940628372613,"[462134, 314944, 202595]"
1,760327,2023-09-30 05:09:51.441140+00:00,t,131206,flow,efferent,131206,2024-01-08 19:46:33.536372+00:00,t,79097217839414238,720575940620156283,"[492890, 187909, 204117]"
2,760328,2023-09-30 05:09:51.441140+00:00,t,138571,flow,intrinsic,138571,2024-01-08 19:46:56.609400+00:00,t,81277136946061019,720575940640372277,"[620140, 99296, 158026]"
3,760329,2023-09-30 05:09:51.441140+00:00,t,137463,flow,intrinsic,137463,2024-01-08 19:46:56.002999+00:00,t,80434498856324372,720575940634601954,"[570818, 205376, 191260]"
4,760330,2023-09-30 05:09:51.441140+00:00,t,133024,flow,intrinsic,133024,2024-01-08 19:46:44.939864+00:00,t,81840568002630413,720575940624155416,"[652270, 128644, 169218]"
...,...,...,...,...,...,...,...,...,...,...,...,...
139250,397994,2023-06-19 23:22:47.293270+00:00,t,104828,flow,intrinsic,104828,2023-06-19 06:48:15.001237+00:00,t,83952592400822837,720575940626100238,"[776911, 183792, 174548]"
139251,436766,2023-06-19 23:24:21.179830+00:00,t,104809,flow,intrinsic,104809,2023-06-19 06:48:14.983796+00:00,t,84798048190170122,720575940626915216,"[824625, 246096, 194229]"
139252,402701,2023-06-19 23:23:00.124836+00:00,t,83362,flow,intrinsic,83362,2023-06-19 06:47:20.709997+00:00,t,75298954786869572,720575940645731620,"[272447, 288010, 156593]"
139253,436758,2023-06-19 23:24:21.174050+00:00,t,104819,flow,intrinsic,104819,2023-06-19 06:48:14.992064+00:00,t,84658822530652061,720575940629075755,"[815896, 335911, 204215]"


In [12]:
projection_test = test_df[(test_df['cell_type'] == 'efferent') | (test_df['cell_type'] == 'afferent')]
display(projection_test)

,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pt_supervoxel_id,pt_root_id,pt_position
0,760326,2023-09-30 05:09:51.441140+00:00,t,135,flow,efferent,135,2023-06-19 06:43:32.694340+00:00,t,78536398189730110,720575940628372613,"[462134, 314944, 202595]"
1,760327,2023-09-30 05:09:51.441140+00:00,t,131206,flow,efferent,131206,2024-01-08 19:46:33.536372+00:00,t,79097217839414238,720575940620156283,"[492890, 187909, 204117]"
5,760331,2023-09-30 05:09:51.441140+00:00,t,3121,flow,efferent,3121,2023-06-19 06:43:36.051994+00:00,t,78958542002158163,720575940608078347,"[484215, 311742, 214255]"
30,760356,2023-09-30 05:09:51.441140+00:00,t,38124,flow,efferent,38124,2023-06-19 06:45:18.928129+00:00,t,80436216910286271,720575940624327572,"[570867, 309554, 210943]"
31,760357,2023-09-30 05:09:51.441140+00:00,t,38130,flow,efferent,38130,2023-06-19 06:45:18.932753+00:00,t,80928866772210317,720575940607641010,"[599591, 311996, 200486]"
...,...,...,...,...,...,...,...,...,...,...,...,...
138698,383394,2023-06-19 23:22:07.945981+00:00,t,32140,flow,efferent,32140,2023-06-19 06:44:57.571775+00:00,t,80505211063621602,720575940625345511,"[573316, 225871, 150815]"
138702,383393,2023-06-19 23:22:07.944715+00:00,t,32144,flow,efferent,32144,2023-06-19 06:44:57.575310+00:00,t,80155360073555253,720575940632412333,"[553163, 344110, 115567]"
138732,383431,2023-06-19 23:22:07.994878+00:00,t,32128,flow,efferent,32128,2023-06-19 06:44:57.561565+00:00,t,79238986052608563,720575940627210568,"[500642, 251331, 176953]"
138737,383392,2023-06-19 23:22:07.943467+00:00,t,32185,flow,efferent,32185,2023-06-19 06:44:57.612244+00:00,t,80575305064017924,720575940633953434,"[577766, 210637, 187695]"


So a more thorough strategy is to use all the classification_systems.  Here's my strategy to get all the relevant neurons by root_id

1. cell_type classification
* Get all the DN neurons
* Get all the VUM neurons
* Get any neurons with AN (I don't think there are any)

2. cell_class classification
* If they are AN, mechanosensory, unknown sensory, gustatory

3. flow classification
* all efferent and afferent types

Then make sure we extract more metadata to provide more information to aid filtering

## Evaluate single neuron codes

### Load Codex information

In [13]:
# Function 1: Get the dataframes
def get_codex_synapse_predictions(root_id, client):
    """
    Function to gather synapse predictions for a given root ID.
    """

    # Pull initial data from the CAVE client
    pre_df = client.materialize.query_table(
        table='synapses_nt_v1',
        filter_in_dict={'pre_pt_root_id': [root_id]},
    )

    return pre_df

# Function 2: Get the neuron skeleton
def get_neuron_skeleton(root_id):
    """
    Function to gather the neuron skeleton for a given root ID.
    """
    neuron_skeleton = flywire.get_skeletons(root_id)

    return neuron_skeleton

# Function 3: Identify NT contributions per neuron
def identify_nt_contributions(pre_df, softmax_threshold=0.25):
    """
    Function to identify neurotransmitter contributions per neuron.
    Inputs:
    pre_df: DataFrame containing synapse predictions with columns for each neurotransmitter type.
    softmax_threshold: Float, the threshold below which a neurotransmitter is considered 'Unknown'.

    Returns:
    nt_counts: a dictionary with counts of each neurotransmitter type.
    synapse_predictions: a numpy array with synapse predictions for each neurotransmitter type. (NB excludes 'Unknown' type)
    """

    #Synapse predictions
    synapse_predictions = np.zeros((len(pre_df), 6), dtype=np.float32)
    for i in range(pre_df.shape[0]):
        row = [pre_df.iloc[i]['gaba'], pre_df.iloc[i]['ach'], pre_df.iloc[i]['glut'], pre_df.iloc[i]['oct'], pre_df.iloc[i]['ser'], pre_df.iloc[i]['da']]
        synapse_predictions[i] = row
        
    # Define neurotransmitter types
    nt_types = ['GABA', 'ACh', 'Glut', 'Oct', 'Ser', 'DA', 'Unknown']
    
    # Initialize counts and filter along softmax values
    nt_counts = {nt: 0 for nt in nt_types}
    softmax_prediction_vals = {nt: [] for nt in nt_types}

    for i in range(pre_df.shape[0]):
        # Determine the synapse type based on the maximum prediction value  
        synapse_type = np.argmax(pre_df.iloc[i][['gaba', 'ach', 'glut', 'oct', 'ser', 'da']].values)
        softmax_val = np.max(pre_df.iloc[i][['gaba', 'ach', 'glut', 'oct', 'ser', 'da']].values)
        if softmax_val < softmax_threshold:
            synapse_type = 6  # Assign 'Unknown' if below threshold
        
        # Add the softmax value to the corresponding neurotransmitter type
        softmax_prediction_vals[nt_types[synapse_type]].append(softmax_val)
        
        # Increment the count for the neurotransmitter type
        nt_counts[nt_types[synapse_type]] += 1
    
    # Convert to ratios
    total_synapses = pre_df.shape[0]
    for nt in nt_counts.keys():
        if total_synapses > 0:
            nt_counts[nt] = nt_counts[nt] / total_synapses
        else:
            nt_counts[nt] = 0
    
    return nt_counts, synapse_predictions

### The ratio calculation function

In [14]:
# Function 4: Predict which NTs the neuron uses via ratios
def predict_neurotransmitter_usage_ratios(nt_used, nt_counts, ratio_threshold):
    """
    Function to predict which neurotransmitters a neuron uses based on ratios.
    Returns a list of neurotransmitter types that exceed the ratio threshold.

    Inputs:
    nt_used: List of boolean values indicating whether each neurotransmitter is used.
    nt_counts: Dictionary with ratio of each neurotransmitter type.
    ratio_threshold: Float, the threshold above which a neurotransmitter is considered used.
    """
    c = 0
    for nt, ratio in nt_counts.items():
        if ratio >= ratio_threshold:
            nt_used[c] = True
        c += 1

    return nt_used

### The clustering function

In [15]:
# Function 5: Mean Shift Clustering
def predict_neurotransmitter_usage_clustering(pre_df, nt_used, synapse_predictions, bandwidth_quantile=1/7, min_synapses_ratio=0.01):
    """
    Function to predict neurotransmitter usage based on clustering.
    Returns a list of neurotransmitter types that are used based on clustering.

    Inputs:
    pre_df: DataFrame containing synapse predictions with columns for each neurotransmitter type.
    nt_used: List of boolean values indicating whether each neurotransmitter is used.
    synapse_predictions: Numpy array with synapse predictions for each neurotransmitter type.
    bandwidth_quantile: Float, the quantile to use for bandwidth estimation.
    min_synapses_ratio: Float, the ratio of minimum synapses per cluster.
    
    Returns:
    nt_used: List of boolean values indicating whether each neurotransmitter is used.
    """
    temp_df = pre_df.copy()
    all_coords = np.array([pos for pos in pre_df['pre_pt_position'].values])
    total_synapses = all_coords.shape[0]

    # Calculate Parameters for mean shift clustering
    bandwidth = estimate_bandwidth(all_coords, quantile=bandwidth_quantile) #Set to 1/the total number of NT types (1/7)
    min_synapses = int(total_synapses * min_synapses_ratio)  # Set to 1% of the total number of synapses
    # print(f"Estimated bandwidth: {bandwidth}")
    # print(f"Minimum synapses per cluster: {min_synapses}")

    # Add the synapse type to the DataFrame
    synapse_ids = []
    for i in range(pre_df.shape[0]):
        #Create a code for the synapse type
        synapse_type = np.argmax(synapse_predictions[i])
        softmax_val = np.max(synapse_predictions[i])
        if softmax_val < 0.25:
            synapse_type = 6
        synapse_ids.append(synapse_type)
    temp_df['synapse_type'] = synapse_ids

    # Loop through each neurotransmitter type and apply mean shift clustering
    for nt_type in range(7):
        nt_df = temp_df[temp_df['synapse_type'] == nt_type]
        coords = np.array([pos for pos in nt_df['pre_pt_position'].values])
        if coords.shape[0] < 2:
            # print(f"Skipping NT type {nt_type} due to insufficient synapses.")
            continue
        cluster_centers, labels = mean_shift(coords, bandwidth=bandwidth)
        
        # Filter clusters with fewer than min_synapses
        unique_labels, counts = np.unique(labels, return_counts=True)
        valid_labels = unique_labels[counts >= min_synapses]        
        n_clusters = len(valid_labels)

        if n_clusters > 0:
            # print(f"NT type {nt_type} has a valid number of clusters:", n_clusters)
            nt_used[nt_type] = True
        else:
            # print(f"NT type {nt_type} has no valid clusters.")
            nt_used[nt_type] = False

    return nt_used

### Final neuron function


In [16]:
def identify_nt_contributions_for_neuron(root_id, 
                                         client, 
                                         softmax_threshold=0.25, 
                                         ratio_threshold=0.16, 
                                         bandwidth_quantile_value=1/7, 
                                         min_synapses_ratio_value=0.01,
                                         display_fig=False)-> dict:
    """
    Function to identify neurotransmitter contributions per neuron.
    Inputs:
    root_id: The root ID of the neuron to analyze.
    client: The CAVE client to use for querying data.
    softmax_threshold: Float, the threshold below which a neurotransmitter is considered 'Unknown'.
    ratio_threshold: Float, the threshold above which a neurotransmitter is considered used.
    bandwidth_quantile_value: Float, the quantile to use for bandwidth estimation in clustering.
    min_synapses_ratio_value: Float, the ratio of minimum synapses required for a cluster to be considered valid.
    display_fig: Boolean, whether to display the figures or not.
    
    Returns a list of neurotransmitter types that exceed the ratio threshold.
    """
    # Get the synapse predictions
    pre_df = get_codex_synapse_predictions(root_id, client)
    
    # Identify neurotransmitter contributions
    nt_counts, synapse_predictions = identify_nt_contributions(pre_df, softmax_threshold)
    
    # Predict neurotransmitter usage based on ratios
    nt_used = np.zeros(7, dtype=bool)
    nt_used = predict_neurotransmitter_usage_ratios(nt_used, nt_counts, ratio_threshold)

    # Predict the neurotransmitter usage via clustering
    # print(bandwidth_quantile_value)
    nt_used = predict_neurotransmitter_usage_clustering(pre_df,
                                                         nt_used,
                                                         synapse_predictions, 
                                                         bandwidth_quantile=bandwidth_quantile_value, 
                                                         min_synapses_ratio=min_synapses_ratio_value)
    
    # Create a output dictionary to store the results
    nt_results = {
        'root_id': root_id,
    }
    # Add the nt_used boolean array to the results with neurotransmitter names as key.
    nt_names = nt_counts.keys()
    for i, nt in enumerate(nt_names):
        nt_results[nt] = nt_used[i]

    if display_fig:
        neuron_skeleton = get_neuron_skeleton(root_id)
        # Plot the neuron skeleton with neurotransmitter contributions
        fig, ax = plt.subplots(2,4, figsize=(20, 10))

        # Plot the synapse predictions as a heatmap
        sns.heatmap(synapse_predictions, ax=ax[0,0])
        ax[0,0].set_title('Synapse Predictions Heatmap')
        ax[0,0].set_xlabel('Neurotransmitter Type')
        ax[0,0].set_xticks(np.arange(6) + 0.5)
        ax[0,0].set_xticklabels(['GABA', 'ACh', 'Glut', 'Oct', 'Ser', 'DA'], rotation=45)

        # Plot each row of the synapse predictions as a line plot
        for i in range(synapse_predictions.shape[0]):
            ax[0,1].plot(synapse_predictions[i], color='k', label=f'Synapse {i+1}', alpha=0.1)
        mean_predictions = np.mean(synapse_predictions, axis=0)
        median_predictions = np.median(synapse_predictions, axis=0)
        ax[0,1].plot(mean_predictions, label='Mean', color='r')
        ax[0,1].plot(median_predictions, label='Median', color='blue')
        ax[0,1].set_xlabel('Neurotransmitter Type')
        ax[0,1].set_xticks([0, 1, 2, 3, 4, 5])
        ax[0,1].set_xticklabels(['GABA', 'ACh', 'Glut', 'Oct', 'Ser', 'DA'], rotation=45)

        # Plot the neuron skeleton with the synapse predictions - all 3 combinations
        navis.plot2d(neuron_skeleton, ax=ax[1,0], color='k', view='xy')
        navis.plot2d(neuron_skeleton, ax=ax[1,1], color='k', view='yz')
        navis.plot2d(neuron_skeleton, ax=ax[1,2], color='k', view='xz')

        # Define neurotransmitter types and colors
        present_types = set()
        softmax_prediction_vals = {
            'GABA': [],
            'ACh': [],
            'Glut': [],
            'Oct': [],
            'Ser': [],
            'DA': [],
            'Unknown': []
        }
        nt_types = ['GABA', 'ACh', 'Glut', 'Oct', 'Ser', 'DA', 'Unknown']
        colours = ['red', 'green', 'blue', 'cyan', 'magenta', 'yellow', 'gray']

        for i in range(synapse_predictions.shape[0]):
            # Determine the synapse type based on the maximum prediction value  
            synapse_type = np.argmax(synapse_predictions[i])
            softmax_val = np.max(synapse_predictions[i])
            if softmax_val < softmax_threshold:
                synapse_type = 6  # Assign 'Unknown' if below threshold
            colour = colours[synapse_type]
            present_types.add(synapse_type)

            # Add the softmax value to the corresponding neurotransmitter type
            softmax_prediction_vals[nt_types[synapse_type]].append(softmax_val)
        
            # Find the position of the pre-synapse
            pos = pre_df.iloc[i]['pre_pt_position']
            pos_xy = (pos[0], pos[1])
            pos_yz = (pos[1], pos[2])
            pos_xz = (pos[0], pos[2])
        
            # Plot the synapse prediction on the neuron skeleton
            ax[1,0].plot(pos_xy[0], pos_xy[1], 'o', color=colour, markersize=2, alpha=0.75)
            ax[1,1].plot(pos_yz[0], pos_yz[1], 'o', color=colour, markersize=2, alpha=0.75)
            ax[1,2].plot(pos_xz[0], pos_xz[1], 'o', color=colour, markersize=2, alpha=0.75)
    
        # Create legend entries only for neurotransmitter types that are present
        legend_elements = []
        for nt_idx in sorted(present_types):
            legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', 
                                            markerfacecolor=colours[nt_idx], markersize=8,
                                            label=nt_types[nt_idx]))
    
        # Place the legend horizontally below the x-axis
        ax[1,0].legend(
            handles=legend_elements,
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            borderaxespad=0.,
            ncol=round(len(legend_elements)/2),
            frameon=False
        )
        ax[1,0].set_title('Neuron Skeleton with Synapse Predictions - Coronal')
        ax[1,1].set_title('Neuron Skeleton with Synapse Predictions - Sagittal')
        ax[1,2].set_title('Neuron Skeleton with Synapse Predictions - Axial')

        # Plot the counts of each neurotransmitter type
        ax[0,3].bar(nt_counts.keys(), nt_counts.values(), color=colours)
        ax[0,3].set_xlabel('Neurotransmitter Type')
        ax[0,3].set_ylabel('Count')
        ax[0,3].set_title('Counts of Neurotransmitter Types')

        # Plot the counts of each neurotransmitter type
        pcts = [nt_counts[nt] / synapse_predictions.shape[0] for nt in nt_counts.keys()]
        ax[1,3].bar(nt_counts.keys(), pcts, color=colours)
        ax[1,3].set_xlabel('Neurotransmitter Type')
        ax[1,3].set_ylabel('Ratios')
        ax[1,3].set_title('Ratios of Neurotransmitter Types')
        
        # Softmax confidence of the predictions
        sns.stripplot(data=softmax_prediction_vals, ax=ax[0,2])
        ax[0,2].set_title('Softmax Confidence of Predictions')
        ax[0,2].set_xlabel('Neurotransmitter Type')
        ax[0,2].set_ylabel('Softmax Confidence of chosen neurotransmitter')

        plt.suptitle(f'Synapse Predictions for Neuron {root_id} with Softmax Threshold {softmax_threshold}', fontsize=16)
        plt.tight_layout()
        plt.show()
    
    return nt_results

### Quick Check to see if everything is working

In [17]:
root_id = "720575940607155890"
nt_results = identify_nt_contributions_for_neuron(root_id, 
                                              client, 
                                              softmax_threshold=0.25, 
                                              ratio_threshold=0.16, 
                                              bandwidth_quantile_value=1/7, 
                                              min_synapses_ratio_value=0.01,
                                              display_fig=False)
print(f"result: {nt_results}")

result: {'root_id': '720575940607155890', 'GABA': np.False_, 'ACh': np.True_, 'Glut': np.True_, 'Oct': np.False_, 'Ser': np.False_, 'DA': np.True_, 'Unknown': np.False_}


## Find neurons of the appropriate cell type
There are three classification systems that I'm going to use. And we also need to get the metadata for the neuron that might be helpful

1. cell_type classification (Use regex expressions)
* Get all the DN neurons
* Get all the VUM neurons
* Get any neurons with AN (I don't think there are any)

2. cell_class classification
* If they are `AN`, `mechanosensory`, `unknown_sensory`, `gustatory`

3. flow classification
* all `efferent` and `afferent` types

In [18]:
# Function to get the root IDs for a given cell type provided it meets the minimum synapse criteria
def get_roots_for_cell_types(client, expression, min_synapses=400)->pd.DataFrame:
    """
    Function to get the root IDs for a given cell type.
    Returns a datafrane of root IDs and associated metadata.

    Inputs:
    client: The CAVE client to use for querying data.
    expression: A regex expression to filter the cell types.
    min_synapses: Integer, the minimum number of synapses required for a root ID to be included in the results.
    """
    # Query the materialize table for the cell type
    roots_df = client.materialize.query_table(
        "hierarchical_neuron_annotations",
        filter_in_dict={'classification_system': ["cell_type"]},
        filter_regex_dict={'cell_type': expression}
    )

    # Now sort the DataFrame by cell_type and filter by num_synapses
    cell_types = roots_df['cell_type'].unique().tolist()
    n_cell_types = len(cell_types)
    print(f"Found {n_cell_types} unique cell types in the hierarchical_neuron_annotations table.")

    output_df = pd.DataFrame(columns=['root_id', 'classification_system', 'cell_type', 'num_synapses'])
    pct = 0
    for c, cell_type in enumerate(cell_types):
        # print(f"Cell type: {cell_type}, {np.round((((c+1)/n_cell_types)*100),2)}% complete")
        temp_df = roots_df[roots_df['cell_type'] == cell_type]
        pct_val = np.round((((c+1)/n_cell_types)*100))
        if pct_val > pct:
            print(f"{pct_val}% complete")
            pct = pct_val
        temp_root_ids = temp_df['pt_root_id'].tolist()
        n_neurons = len(temp_root_ids)
        # if n_neurons > 10 then we have concatenate to speed up the query
        if n_neurons <= 10:
            stats_df = client.materialize.query_table(
                table='synapses_nt_v1',
                filter_in_dict={'pre_pt_root_id': temp_root_ids}
                )
            # Now count the number of synapses per root
            check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

            root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
            num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

            final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
            final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

            type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
            type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

            output_df = pd.concat([output_df, type_df], ignore_index=True)
        else:
            # Loop through the root IDs in chunks of 10
            for i in range(0, n_neurons, 10):
                temp_root_ids_chunk = temp_root_ids[i:i+10]
                stats_df = client.materialize.query_table(
                    table='synapses_nt_v1',
                    filter_in_dict={'pre_pt_root_id': temp_root_ids_chunk}
                )
                # Now count the number of synapses per root
                check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

                root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
                num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

                final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
                final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

                type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
                type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

                output_df = pd.concat([output_df, type_df], ignore_index=True)
        c += 1

    return output_df

def get_roots_for_cell_class(client, specific_class, min_synapses=400)->dict:
    """
    Function to get the root IDs for a given cell type.
    Returns a dictionary of root IDs and the number of synapses.

    Inputs:
    client: The CAVE client to use for querying data.
    specific_class: The specific cell class to filter the root IDs by.
    min_synapses: Integer, the minimum number of synapses required for a root ID to be included in the results.
    """
    # Query the materialize table for the cell type
    roots_df = client.materialize.query_table(
        "hierarchical_neuron_annotations",
        filter_in_dict={'classification_system': ["cell_class"],
                        'cell_type': [specific_class]},
    )
    # Now sort the DataFrame by cell_type and filter by num_synapses
    cell_types = roots_df['cell_type'].unique().tolist()
    n_cell_types = len(cell_types)
    print(f"Found {n_cell_types} unique cell types in the hierarchical_neuron_annotations table.")
    
    pct = 0
    output_df = pd.DataFrame(columns=['root_id', 'classification_system', 'cell_type', 'num_synapses'])
    for c, cell_type in enumerate(cell_types):
        # print(f"Cell type: {cell_type}, {np.round((((c+1)/n_cell_types)*100),2)}% complete")
        temp_df = roots_df[roots_df['cell_type'] == cell_type]
        temp_root_ids = temp_df['pt_root_id'].tolist()
        n_neurons = len(temp_root_ids)

        pct_val = np.round((((c+1)/n_cell_types)*100),2)
        if pct_val > pct:
            print(f"{pct_val}% complete")
            pct = pct_val
        
        # if n_neurons > 10 then we have concatenate to speed up the query and avoid the CAVE timeout/synapse limit
        if n_neurons <= 10:
            stats_df = client.materialize.query_table(
                table='synapses_nt_v1',
                filter_in_dict={'pre_pt_root_id': temp_root_ids}
                )
            # Now count the number of synapses per root
            check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

            root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
            num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

            final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
            final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

            type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
            type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

            output_df = pd.concat([output_df, type_df], ignore_index=True)
        else:
            # Loop through the root IDs in chunks of 10
            print(f"Having to loop through the root IDs in chunks of 10 due to the number of neurons ({n_neurons}) being greater than 10.")
            for i in range(0, n_neurons, 10):
                # print(f"Evaluating chunk {i//10 + 1} of {n_neurons//10 + 1}.")
                temp_root_ids_chunk = temp_root_ids[i:i+10]
                stats_df = client.materialize.query_table(
                    table='synapses_nt_v1',
                    filter_in_dict={'pre_pt_root_id': temp_root_ids_chunk}
                )
                # Now count the number of synapses per root
                check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

                root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
                num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

                final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
                final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

                type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
                type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

                output_df = pd.concat([output_df, type_df], ignore_index=True)
        c += 1

    return output_df

def get_roots_for_cell_flow(client, specific_flow, min_synapses=400)->dict:
    """
    Function to get the root IDs for a given cell type.
    Returns a dictionary of root IDs and the number of synapses.

    Inputs:
    client: The CAVE client to use for querying data.
    specific_flow: The specific cell class to filter the root IDs by.
    min_synapses: Integer, the minimum number of synapses required for a root ID to be included in the results.
    """
    # Query the materialize table for the cell type
    roots_df = client.materialize.query_table(
        "hierarchical_neuron_annotations",
        filter_in_dict={'classification_system': ["flow"],
                        'cell_type': [specific_flow]},
    )
    # Now sort the DataFrame by cell_type and filter by num_synapses
    cell_types = roots_df['cell_type'].unique().tolist()
    n_cell_types = len(cell_types)
    print(f"Found {n_cell_types} unique cell types in the hierarchical_neuron_annotations table.")
    pct = 0
    output_df = pd.DataFrame(columns=['root_id', 'classification_system', 'cell_type', 'num_synapses'])

    for c, cell_type in enumerate(cell_types):
        # print(f"Cell type: {cell_type}, {np.round((((c+1)/n_cell_types)*100),2)}% complete")
        pct_val = np.round((((c+1)/n_cell_types)*100))
        if pct_val > pct:
            print(f"{pct_val}% complete")
            pct = pct_val

        temp_df = roots_df[roots_df['cell_type'] == cell_type]
        temp_root_ids = temp_df['pt_root_id'].tolist()
        n_neurons = len(temp_root_ids)
        print(f"There are {n_neurons} neurons for the cell type {cell_type}.")
        # if n_neurons > 10 then we have concatenate to speed up the query
        if n_neurons <= 10:
            stats_df = client.materialize.query_table(
                table='synapses_nt_v1',
                filter_in_dict={'pre_pt_root_id': temp_root_ids}
                )
            # Now count the number of synapses per root
            check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

            root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
            num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

            final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
            final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

            type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
            type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

            output_df = pd.concat([output_df, type_df], ignore_index=True)
        else:
            print(f"Having to loop through the root IDs in chunks of 10 due to the number of neurons ({n_neurons}) being greater than 10.")
            # Loop through the root IDs in chunks of 10
            for i in range(0, n_neurons, 10):
                # print(f"Chunk {i//10 + 1} of {n_neurons//10 + 1}.")
                temp_root_ids_chunk = temp_root_ids[i:i+10]
                stats_df = client.materialize.query_table(
                    table='synapses_nt_v1',
                    filter_in_dict={'pre_pt_root_id': temp_root_ids_chunk}
                )
                # Now count the number of synapses per root
                check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

                root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
                num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

                final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
                final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

                type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
                type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

                output_df = pd.concat([output_df, type_df], ignore_index=True)
        c += 1

    return output_df

# Set the timeout feature
class Timeout(Exception):
    pass

def _raise_timeout(signum, frame):
    raise Timeout("identify_nt_contributions_for_neuron timed out")

# Now loop and append the results for each neuron
def evaluate_all_neurons_for_classifier_cell_type(client, 
                                       classifier, 
                                       cell_type, 
                                       softmax_threshold=0.25, 
                                       ratio_threshold=0.16, 
                                       bandwidth_quantile_value=1/7, 
                                       min_synapses_ratio_value=0.01, 
                                       min_synapses=400, 
                                       display_fig=False,
                                       timeout_seconds=900)-> tuple [list, list]:
    """
    Function to evaluate all neurons for a given cell type.
    Inputs:
    client: The CAVE client to use for querying data.
    classifier: The classifier to use for the cell type. There are three options: 'cell_type', 'cell_class', 'flow'.
    cell_type: The cell type to evaluate.
    softmax_threshold: Float, the threshold below which a neurotransmitter is considered 'Unknown'.
    ratio_threshold: Float, the threshold above which a neurotransmitter is considered used.
    bandwidth_quantile_value: Float, the quantile to use for bandwidth estimation in clustering.
    min_synapses_ratio_value: Float, the ratio of minimum synapses required for a cluster to be considered valid.
    min_synapses: Integer, the minimum number of synapses required for a root ID to be included in the results.
    display_fig: Boolean, whether to display the figure.
    timeout_seconds: Integer, the maximum time in seconds to allow for processing each neuron.

    Returns a tuple of two lists, containing dictionaries with neurotransmitter contributions for each neuron and root_ids which were timed out
    """
    # Check the classifier and get the root IDs
    if classifier == 'cell_type':
        print("Evaluating cell type classifier.")
        root_id_df = get_roots_for_cell_types(client, cell_type, min_synapses)
    elif classifier == 'cell_class':
        print("Evaluating cell class classifier.")
        root_id_df = get_roots_for_cell_class(client, cell_type, min_synapses)
    elif classifier == 'flow':
        print("Evaluating flow classifier.")
        root_id_df = get_roots_for_cell_flow(client, cell_type, min_synapses)
    else:
        raise ValueError(f"Unknown classifier: {classifier}")

    print(f"Found {len(root_id_df)} root IDs for cell type {cell_type} with at least {min_synapses} synapses.")
    nt_results = []
    timeout_results = []
    # Return nothing if no root IDs found
    if root_id_df.empty:
        print("No root IDs found for the given cell type and classifier.")
        return nt_results, timeout_results
    c = 0
    total_neurons = len(root_id_df)
    pct = 0
    for index, row in root_id_df.iterrows():
        root_id = row['root_id']
        num_synapses = row['num_synapses']
        pct_val = np.round((((c+1)/total_neurons)*100))
        if pct_val > pct:
            print(f"{pct_val}% complete")
            pct = pct_val
        # print(f"Processing root ID {root_id} with {num_synapses} synapses. {np.round((((c+1)/total_neurons)*100),2)}% complete")
        # Get the metadata for the root ID
        metadata_dict = {
            'root_id': root_id,
            'classification_system': row['classification_system'],
            'cell_type': row['cell_type'],
            'num_synapses': num_synapses
        }
        # Add a timer error so if a neuron takes more than `timeout_seconds` to process, it will store the metadata in a separate datafile and continue processing the next neuron
        try:
            signal.alarm(timeout_seconds) #starts the timer

            # Identify neurotransmitter contributions for the neuron
            nt_result = identify_nt_contributions_for_neuron(root_id, 
                                                            client, 
                                                            softmax_threshold, 
                                                            ratio_threshold, 
                                                            bandwidth_quantile_value, 
                                                            min_synapses_ratio_value,
                                                            display_fig)
            # Add nt_result and settings to the metadata_dict
            metadata_dict['num_synapses'] = num_synapses
            metadata_dict['softmax_threshold'] = softmax_threshold
            metadata_dict['ratio_threshold'] = ratio_threshold
            metadata_dict['bandwidth_quantile_value'] = bandwidth_quantile_value
            metadata_dict['min_synapses_ratio_value'] = min_synapses_ratio_value

            for key, value in nt_result.items():
                metadata_dict[key] = value
    
            # Append the result to the list
            nt_results.append(metadata_dict)
        except Timeout:
            print(f"Processing of root ID {root_id} timed out. Saving details to timeout_results")
            timeout_results.append(metadata_dict)

        c += 1

    return nt_results, timeout_results

### Intermediary test to see if this works

#### Test cell_type

In [19]:
cell_type = "DNg01_b"
classifier = "cell_type"
nt_results, timeout_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                  classifier, 
                                                  cell_type, 
                                                  softmax_threshold=0.25, 
                                                  ratio_threshold=0.16, 
                                                  bandwidth_quantile_value=1/7, 
                                                  min_synapses_ratio_value=0.01,
                                                  display_fig=False)
print(f"NT results: {(pd.DataFrame(nt_results))}")
print(f"Timeout results: {(pd.DataFrame(timeout_results))}")


Evaluating cell type classifier.
Found 1 unique cell types in the hierarchical_neuron_annotations table.
100.0% complete
Found 1 root IDs for cell type DNg01_b with at least 400 synapses.
100.0% complete
NT results:               root_id classification_system cell_type  num_synapses  \
0  720575940628582607             cell_type   DNg01_b           596   

   softmax_threshold  ratio_threshold  bandwidth_quantile_value  \
0               0.25             0.16                  0.142857   

   min_synapses_ratio_value  GABA   ACh  Glut    Oct   Ser    DA  Unknown  
0                      0.01  True  True  True  False  True  True    False  
Timeout results: Empty DataFrame
Columns: []
Index: []


#### Test cell_class

In [ ]:
cell_type = "unknown_sensory"
classifier = "cell_class"
nt_results, timeout_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                  classifier, 
                                                  cell_type, 
                                                  softmax_threshold=0.25, 
                                                  ratio_threshold=0.16, 
                                                  bandwidth_quantile_value=1/7, 
                                                  min_synapses_ratio_value=0.01,
                                                  display_fig=False)
print(f"result: {pd.DataFrame(nt_results)}")

#### Test cell_flow

In [ ]:
cell_type = "efferent"
classifier = "flow"
nt_results, timeout_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                  classifier, 
                                                  cell_type, 
                                                  softmax_threshold=0.25, 
                                                  ratio_threshold=0.16, 
                                                  bandwidth_quantile_value=1/7, 
                                                  min_synapses_ratio_value=0.01,
                                                  display_fig=False)
print(f"nt_result: {pd.DataFrame(nt_results)}")
print(f"timeout_result: {pd.DataFrame(timeout_results)}")


## Actual Investigation

### Just test 3 cell types for a trial

In [ ]:
classifier = "cell_type"
cell_types = ["DNg01_a", "DNg01_b", "DNg02_a"] # List of cell types to evaluate
eval_results = []
all_timeouts = []
for cell_type in cell_types:
    nt_results,timeout_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                  classifier, 
                                                  cell_type, 
                                                  softmax_threshold=0.25, 
                                                  ratio_threshold=0.16, 
                                                  bandwidth_quantile_value=1/7, 
                                                  min_synapses_ratio_value=0.01,
                                                  display_fig=False)
    eval_results.extend(nt_results)
    all_timeouts.extend(timeout_results)

# Convert the results to a DataFrame for better visualization
eval_df = pd.DataFrame(eval_results)
all_timeouts_df = pd.DataFrame(all_timeouts)

# Now evaluate for co-transission (>1 NT type true)
eval_df['co_transmission'] = eval_df.apply(lambda row: sum(row[nt] for nt in ['GABA', 'ACh', 'Glut', 'Oct', 'Ser', 'DA']) > 1, axis=1)
# Display the DataFrame
display(eval_df)

Evaluating cell type classifier.
Found 1 unique cell types in the hierarchical_neuron_annotations table.
100.0% complete
Found 1 root IDs for cell type DNg01_a with at least 400 synapses.
100.0% complete
Evaluating cell type classifier.
Found 1 unique cell types in the hierarchical_neuron_annotations table.
100.0% complete
Found 1 root IDs for cell type DNg01_b with at least 400 synapses.
100.0% complete
Evaluating cell type classifier.
Found 1 unique cell types in the hierarchical_neuron_annotations table.
100.0% complete
Found 1 root IDs for cell type DNg02_a with at least 400 synapses.
100.0% complete


,root_id,classification_system,cell_type,num_synapses,softmax_threshold,ratio_threshold,bandwidth_quantile_value,min_synapses_ratio_value,GABA,ACh,Glut,Oct,Ser,DA,Unknown,co_transmission
0,720575940622925972,cell_type,DNg01_a,421,0.25,0.16,0.142857,0.01,True,True,True,False,True,True,False,True
1,720575940628582607,cell_type,DNg01_b,596,0.25,0.16,0.142857,0.01,True,True,True,False,True,True,False,True
2,720575940610863310,cell_type,DNg02_a,617,0.25,0.16,0.142857,0.01,True,True,True,False,True,True,False,True


: 

## Full investigation

In [ ]:
# Search by cell types
classifier = "cell_type"
cell_types = [
    # "DNa.*", 
    # "DNb.*",
    # "DNc.*",
    # "DNd.*",
    # "DNg0.*",
    # "DNg1.*",
    # "DNg2.*",
    # "DNg3.*",
    # "DNg4.*",
    # "DNg5.*",
    # "DNg6.*",
    # "DNg7.*",
    # "DNg8.*",
    # "DNg9.*",
    # "DNp0.*",
    # "DNp1.*",
    # "DNp2.*",
    # "DNp3.*",
    # "DNp4.*",
    # "DNp5.*",
    # "DNp6.*",
    # "DNp7.*",
    # "DNp8.*",
    # "DNp9.*",
    "DNp[a-g].*",
    "DNp[h-l].*",
    # "DNp[m-z].*",
    "DNx.*",
    ".*VUM.*",
    ".*AN.*"
]
combined_df = []
combined_timeouts_df = []
for cell_type in cell_types:
    print(f"Evaluating cell type: {cell_type}")
    nt_results, timeout_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                    classifier, 
                                                    cell_type, 
                                                    softmax_threshold=0.25, 
                                                    ratio_threshold=0.16, 
                                                    bandwidth_quantile_value=1/7, 
                                                    min_synapses_ratio_value=0.01,
                                                    display_fig=False)
    # nt_results dictionary is not empty
    if not nt_results:
        intermediary_csv_name = f"nt_results_cell-type_{cell_type}.csv"
        print(f"Saving results to {intermediary_csv_name}")
        pd.DataFrame(nt_results).to_csv(intermediary_csv_name, index=False)
        combined_df.extend(nt_results)
    if not timeout_results:
        combined_timeouts_df.extend(timeout_results)

# Search by cell class
classifier = "cell_class"
cell_types = ["AN", "mechanosensory", "unknown_sensory", "gustatory"]

for cell_type in cell_types:
    print(f"Evaluating cell class: {cell_type}")
    nt_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                    classifier, 
                                                    cell_type, 
                                                    softmax_threshold=0.25, 
                                                    ratio_threshold=0.16, 
                                                    bandwidth_quantile_value=1/7, 
                                                    min_synapses_ratio_value=0.01,
                                                    display_fig=False)
    # nt_results dictionary is not empty
    if not nt_results:
        intermediary_csv_name = f"nt_results_cell-type_{cell_type}.csv"
        print(f"Saving results to {intermediary_csv_name}")
        pd.DataFrame(nt_results).to_csv(intermediary_csv_name, index=False)
        combined_df.extend(nt_results)
    if not timeout_results:
        combined_timeouts_df.extend(timeout_results)

# Search by flow
classifier = "flow"
cell_types = ["efferent", "afferent"]
for cell_type in cell_types:
    print(f"Evaluating flow: {cell_type}")
    nt_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                    classifier, 
                                                    cell_type, 
                                                    softmax_threshold=0.25, 
                                                    ratio_threshold=0.16, 
                                                    bandwidth_quantile_value=1/7, 
                                                    min_synapses_ratio_value=0.01,
                                                    display_fig=False)
    # nt_results dictionary is not empty
    if not nt_results:
        intermediary_csv_name = f"nt_results_cell-type_{cell_type}.csv"
        print(f"Saving results to {intermediary_csv_name}")
        pd.DataFrame(nt_results).to_csv(intermediary_csv_name, index=False)
        combined_df.extend(nt_results)
    if not timeout_results:
        combined_timeouts_df.extend(timeout_results)


Evaluating cell type: DNp[a-l].*
Evaluating cell type classifier.
Found 56 unique cell types in the hierarchical_neuron_annotations table.
2.0% complete
4.0% complete
5.0% complete
7.0% complete
9.0% complete
11.0% complete
12.0% complete
14.0% complete
16.0% complete
18.0% complete
20.0% complete
21.0% complete
23.0% complete
25.0% complete
27.0% complete
29.0% complete
30.0% complete
32.0% complete
34.0% complete
36.0% complete
38.0% complete
39.0% complete
41.0% complete
43.0% complete
45.0% complete
46.0% complete
48.0% complete
50.0% complete
52.0% complete
54.0% complete
55.0% complete
57.0% complete
59.0% complete
61.0% complete
62.0% complete
64.0% complete
66.0% complete
68.0% complete
70.0% complete
71.0% complete
73.0% complete
75.0% complete
77.0% complete
79.0% complete
80.0% complete
82.0% complete
84.0% complete
86.0% complete
88.0% complete
89.0% complete
91.0% complete
93.0% complete
95.0% complete
96.0% complete
98.0% complete
100.0% complete
Found 115 root IDs for ce

/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


12.0% complete
13.0% complete
14.0% complete
15.0% complete
16.0% complete
17.0% complete
18.0% complete
19.0% complete
20.0% complete
21.0% complete
22.0% complete
23.0% complete
24.0% complete
25.0% complete
26.0% complete
27.0% complete
28.0% complete
29.0% complete
30.0% complete


/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


31.0% complete
32.0% complete
33.0% complete
34.0% complete
35.0% complete
36.0% complete
37.0% complete
38.0% complete
39.0% complete
40.0% complete
41.0% complete
42.0% complete
43.0% complete
44.0% complete
45.0% complete
46.0% complete


...

25.0% complete


/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


38.0% complete
50.0% complete
62.0% complete
75.0% complete
88.0% complete
100.0% complete


/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Evaluating cell type: .*AN.*
Evaluating cell type classifier.
Found 0 unique cell types in the hierarchical_neuron_annotations table.
Found 0 root IDs for cell type .*AN.* with at least 400 synapses.
No root IDs found for the given cell type and classifier.
Saving results to nt_results_cell-type_.*AN.*.csv
Evaluating cell class: AN
Evaluating cell class classifier.
Found 1 unique cell types in the hierarchical_neuron_annotations table.
100.0% complete
Having to loop through the root IDs in chunks of 10 due to the number of neurons (2362) being greater than 10.
